<a href="https://colab.research.google.com/github/rayaguilos06/flyrank-ml-internship/blob/main/w03_data_contract_Aguilos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Install DuckDB if needed
!pip -q install duckdb

import duckdb
from google.colab import userdata

# Read the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Register Hugging Face token
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

# Dataset location
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("✅ Connection successful!")

✅ Connection successful!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# ==========================================================
# 1. Unit of Analysis + Time Window
#
# Unit of Analysis:
# One row represents the daily search performance of one
# content page (content_hash_id) for one client
# (client_hash_id) on a specific report_date.
#
# Time Window:
# March 2026 (month = '2026-03')
#
# Reason:
# A mid-panel month is used to avoid using the final month
# of the dataset during exploration.
# ==========================================================

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,unique_clients,start_date,end_date
0,9841378,331437,55,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# ==========================================================
# 2. Fields: Feature / Label / Context / Excluded
#
# Features
# - gsc_impressions
# - gsc_clicks
# - gsc_avg_position
# - ga4_pageviews
# - ga4_engaged_sessions
#
# Label / Proxy
# Priority score for deciding which pages should be reviewed.
#
# Context
# - report_date
# - client_hash_id
# - content_hash_id
#
# Excluded
# AI referral columns (ai_chatgpt, ai_claude, etc.)
# because they are outside the scope of my lane.
# ==========================================================

con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# ==========================================================
# 3. Verification Queries
#
# Query 1 - Verify the grain
# ==========================================================

display(
    con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month='2026-03'
    LIMIT 5
    """).df()
)

# ==========================================================
# Query 2 - Verify row count and date span
# ==========================================================

display(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day
    FROM {TABLES['fact_daily']}
    WHERE month='2026-03'
    """).df()
)

# ==========================================================
# Query 3 - Availability check
# ==========================================================

display(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(gsc_data_available IS TRUE) AS gsc_available,
        SUM(ga4_data_available IS TRUE) AS ga4_available
    FROM {TABLES['fact_daily']}
    WHERE month='2026-03'
    """).df()
)

,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# ==========================================================
# 4. Data Limits
#
# Limitation:
# This dataset contains anonymized search performance data.
# It provides observed search signals but does not reveal
# Google's ranking algorithm or causal relationships.
# The analysis supports prioritization decisions rather than
# predicting search engine behavior.
# ==========================================================

print("""
Data Limitations

• Uses anonymized search performance data.

• Does not reveal Google's ranking algorithm.

• Shows observed relationships rather than cause-and-effect.

• Results are intended for decision support.

• Performance may change as new search data becomes available.
""")


Data Limitations

• Uses anonymized search performance data.

• Does not reveal Google's ranking algorithm.

• Shows observed relationships rather than cause-and-effect.

• Results are intended for decision support.

• Performance may change as new search data becomes available.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.